# 03 Bear Recovery Scenarios

Defines bear drawdowns and recovery assumptions, then compares the assumed recovery with the recovery required to match selling today.

**Plain English:**
This notebook tests many market drops and asks whether the assumed rebound from each low is enough.

**This answers the question:** "If the market falls by X%, how much rebound do I need, and is my assumed rebound enough?"

Example:
If the market drops 30%, the bear low is 245 from a 350 starting price. The notebook compares the assumed rebound from 245 against the rebound needed to match selling today.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from tax_risk_sim import (
    build_after_tax_sensitivity,
    build_bear_recovery_cases,
    build_bear_recovery_table,
    build_probability_weighted_scenarios,
    build_stop_benchmark,
    compare_stop_reentry_vs_hold,
    sell_today_baseline,
)

pd.options.display.float_format = "{:,.2f}".format

## Inputs

These inputs define the position and the bear-market scenario grid.

**Plain English:**
They say what you own and which market drops to simulate.

**This answers the question:** "Which bear-market drops should be included?"

Example:
Start -5%, end -60%, step -1% creates scenarios for -5%, -6%, -7%, ... -60%.

### Input Explanations

`bear_drawdown_start`, `bear_drawdown_end`, and `bear_drawdown_step` define the bear-market drawdown grid.

**Plain English:**
These create the list of market drops to test.

**This answers the question:** "How many bear scenarios should I generate?"

Example:
-5% to -60% in -1% steps gives 56 bear scenarios.

`recovery_multiplier` inside `build_bear_recovery_cases` controls the assumed rebound from each bear low.

**Plain English:**
This says how strong the bounce is after a fall.

**This answers the question:** "How much recovery am I assuming after each drawdown?"

Example:
With a 1.50 multiplier, a 30% drawdown assumes a 45% recovery from the low.

`recovery_probability` is generated as a simple declining function of drawdown depth. It is an assumption, not a market forecast.

**Plain English:**
Deeper crashes are assumed to have lower chances of recovering in this simple model.

**This answers the question:** "How likely is the assumed rebound in each scenario?"

Example:
A 20% drawdown may get a higher recovery probability than a 50% drawdown.

In [ ]:
from inputs import (
    shares,
    current_price,
    cost_basis_per_share,
    capital_gains_tax_rate,
    stop_loss_drops,
    bear_drawdown_start,
    bear_drawdown_end,
    bear_drawdown_step,
    bear_recovery_multiplier,
    bear_min_recovery_return,
    bear_max_recovery_return,
    bear_base_recovery_probability,
    bear_min_recovery_probability,
)

bear_recovery_cases = build_bear_recovery_cases(
    start=bear_drawdown_start,
    end=bear_drawdown_end,
    step=bear_drawdown_step,
    recovery_multiplier=bear_recovery_multiplier,
    min_recovery_return=bear_min_recovery_return,
    max_recovery_return=bear_max_recovery_return,
    base_recovery_probability=bear_base_recovery_probability,
    min_recovery_probability=bear_min_recovery_probability,
)

bear_recovery_cases

In [ ]:
pd.Series(
    {
        "Shares held": shares,
        "Current price": f"${current_price:,.2f}",
        "Cost basis per share": f"${cost_basis_per_share:,.2f}",
        "Capital gains tax rate": f"{capital_gains_tax_rate:.0%}",
        "Stop-loss levels tested": ", ".join(f"{d:.0%}" for d in stop_loss_drops),
        "Bear drawdown range": (
            f"{bear_drawdown_start:.0%} to {bear_drawdown_end:.0%}"
            f" in {bear_drawdown_step:.0%} steps"
        ),
        "Recovery multiplier": bear_recovery_multiplier,
        "Recovery return range": (
            f"{bear_min_recovery_return:.0%} to {bear_max_recovery_return:.0%}"
        ),
        "Recovery probability range": (
            f"{bear_min_recovery_probability:.0%} to {bear_base_recovery_probability:.0%}"
        ),
    },
    name="Inputs",
)

## Bear Recovery Table

Calculates the bear low, assumed recovery price, expected recovery value, and required recovery to match selling today.

**Plain English:**
For each market drop, this table shows the low price, the possible bounce, and whether waiting looks better than selling today.

**This answers the question:** "If I hold through this bear market and hope for recovery, what outcome am I assuming?"

Example:
For a -30% drawdown from 350, the low is 245. If recovery is 45%, the recovery price is 355.25.

In [ ]:
baseline = sell_today_baseline(
    shares,
    current_price,
    cost_basis_per_share,
    capital_gains_tax_rate,
)
bear_recovery_df = build_bear_recovery_table(
    bear_recovery_cases,
    shares,
    current_price,
    cost_basis_per_share,
    capital_gains_tax_rate,
)
bear_case_order = bear_recovery_df.sort_values("drawdown", ascending=False)["case"].tolist()

bear_recovery_df

### Bear Recovery Column Explanations

`drawdown` is the bear-market drop from today's price.

**Plain English:**
This is how far the price falls.

**This answers the question:** "How bad is the bear scenario?"

Example:
`-0.30` means a 30% drop.

`drawdown_price` is the price at the bear low.

**Plain English:**
This is the low price after the drop.

**This answers the question:** "What price does the drop reach?"

Example:
350 after a 30% drop is 245.

`recovery_return_from_low` is the assumed rebound from that low if recovery happens.

**Plain English:**
This is the bounce from the low.

**This answers the question:** "How strong is the assumed rebound?"

Example:
A 45% recovery from 245 reaches 355.25.

`recovery_price` is the modeled price after the rebound.

**Plain English:**
This is where price ends after the bounce.

**This answers the question:** "What final price am I comparing?"

Example:
245 with a 45% recovery becomes 355.25.

`expected_after_tax_value_if_hold_for_recovery` probability-weights recovery versus no recovery for that bear scenario.

**Plain English:**
This blends the recovery outcome and no-recovery outcome using the assumed chance of recovery.

**This answers the question:** "On average, under this scenario's recovery chance, what is holding worth?"

Example:
If recovery has a 40% chance, the expected value uses 40% recovery value and 60% low-price value.

`required_recovery_return_from_low_to_match_selling_today` is the rebound needed from the bear low to equal selling today after tax.

**Plain English:**
This is the comeback needed just to catch up to selling now.

**This answers the question:** "How much must the price bounce from the low before holding is as good as selling today?"

Example:
If the low is 245 and the target price is 350, the required rebound is about 43%.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5.5))
x = bear_recovery_df["drawdown"].abs() * 100
required = bear_recovery_df["required_recovery_return_from_low_to_match_selling_today"] * 100
assumed = bear_recovery_df["recovery_return_from_low"] * 100
ax.plot(x, required, marker="o", label="Required recovery to match selling today")
ax.plot(x, assumed, marker="o", label="Assumed recovery if recovery happens")
ax.set_title("Bear Drawdown: Required vs Assumed Recovery")
ax.set_xlabel("Bear drawdown from today's price (%)")
ax.set_ylabel("Recovery from drawdown low (%)")
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()